In [ ]:
import os
import argparse
import datetime
import random

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_sched
from torch.utils.data import random_split
# from torch.utils.tensorboard import SummaryWriter

from lib.datasets.utils import get_train_and_val_datasets
from lib.models.my_arcface import MyArcFace
from lib.utils_functions import seed_worker
import pandas as pd
from sklearn.model_selection import train_test_split
def parse_args():
    parser = argparse.ArgumentParser(description='PyTorch Modernized GestaltMatcher')

    # Run parameters
    parser.add_argument('--session', default= 1, type=int, dest='session',
                        help='session used to distinguish model tests.')
    parser.add_argument('--batch_size', type=int, default=128, metavar='BN',
                        help='input batch size for training (default: 280)')
    parser.add_argument('--epochs', type=int, default=1, metavar='E',
                        help='number of epochs to train (default: 500)')
    parser.add_argument('--lr', type=float, default=5e-3, metavar='LR',  # lr=1e-3
                        help='learning rate (default: 0.005)')
    parser.add_argument('--seed', type=int, default=11, metavar='S',
                        help='random seed (default: 11)')
    parser.add_argument('--no_cuda', action='store_true', default=False,
                        help='disables CUDA training')
    parser.add_argument('--log_interval', type=int, default=1000,
                        help='how many images (not batches) to wait before logging training status')
    parser.add_argument('--val_interval', type=int, default=10000,
                        help='how many images (not batches) to wait before validation is evaluated (and optimizer is stepped).')
    parser.add_argument('--use_tensorboard', action='store_true', default=False,
                        help='Use tensorboard for logging')

    # Model parameters
    parser.add_argument('--model_type', default='glint360k_r50', dest='model_type',
                        help='model backend to use')
    parser.add_argument('--in_channels', default=3, dest='in_channels', type=int,
                        help='number of color channels of the images used as input (default: 1)')
    parser.add_argument('--img_size', default=112, dest='img_size', type=int,
                        help='input image size of the model (default: 100)')
    parser.add_argument('--unfreeze', action='store_false', default=True, dest='freeze',
                        help='flag to set if you want to unfreeze the base model weights.')

    # Dataset parameters
    parser.add_argument('--dataset', default='gmdb', dest='dataset',
                        help='which dataset to use. (Options: "casia", "gmdb")')
    parser.add_argument('--dataset_type', default='', dest='dataset_type',
                        help='type of the dataset to use, e.g. normal (="") or augmented(="aug") (default="")')
    parser.add_argument('--dataset_version', default='v1.1.0', dest='dataset_version', type=str,
                        help='version of the dataset to use (default="v1.0.3")')
    parser.add_argument('--lookup_table', default='', dest='lookup_table_path',
                        help='lookup table path, use if you want to load path instead of generation a lookup table (default = "")')

    # File locations
    parser.add_argument('--data_dir', default='data', dest='data_dir',
                        help='Location of the data directory (not dataset). (default = home pc)')
    parser.add_argument('--weight_dir', default='saved_models', dest='weight_dir',
                        help='Location of the model weights directory. (default = "saved_models")')

    # running on my local machine means different path types, and num_workers
    parser.add_argument('--local', action='store_true', default=False,
                        help='Running on local machine, fewer num_workers')

    #
    parser.add_argument('--paper_model', default='a', dest='paper_model', type=str,
                        help='Use when reproducing paper models a) r50-mix, or b) r100')

    return parser.parse_known_args()

In [2]:

args, unknown = parse_args()

use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"Using {'GPU.' if use_cuda else 'CPU, as was explicitly requested, or as GPU is not available.'}")

# Seed everything
np.random.seed(args.seed)
torch.manual_seed(args.seed)
random.seed(args.seed)

# When using GPU we need to set extra seeds
if use_cuda:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.manual_seed_all(args.seed)
    torch.cuda.manual_seed(args.seed)
# torch.set_deterministic(True)

f_wd = c_wd = lr = c_lr = -1
if args.paper_model == 'a':    # r50 mix
    args.model_type = 'glint360k_r50'
    args.batch_size = 64
    f_wd = 5e-5
    c_wd = 5e-4
    lr = 5e-4
    c_lr = 1e-3
elif args.paper_model == 'b':  # r100
    args.model_type = 'glint360k_r100'
    args.batch_size = 128
    f_wd = 0.
    c_wd = 5e-4
    lr = 1e-3
    c_lr = 1e-3

# Dataset and dataloaders
kwargs = {}
if use_cuda:
    kwargs.update({'num_workers': (0 if args.local else 16), 'pin_memory': True})

dataset_train = dataset_val = None
lookup_table = None
distr = None
args.val_bs = 1

# Create and get the training and validation datasets
dataset_train, dataset_val = get_train_and_val_datasets(args.dataset, args.dataset_type, args.dataset_version,
                                                        args.img_size, args.in_channels, args.data_dir,
                                                        train_path=os.path.join(args.data_dir, "GestaltMatcherDB", 
                                                                                args.dataset_version,"gmdb_metadata",f"gmdb_train_images_{args.dataset_version}.csv"),
                                                        val_path=os.path.join(args.data_dir, "GestaltMatcherDB", 
                                                                                args.dataset_version,"gmdb_metadata",f"gmdb_val_images_{args.dataset_version}.csv"),
                                                        img_postfix='_aligned')
    
        # Get the number of classes from the dataset
args.num_classes = dataset_train.get_num_classes()

# Create dataloaders
train_loader = torch.utils.data.DataLoader(dataset_train, **kwargs, shuffle=True, batch_size=args.batch_size,
                                            worker_init_fn=seed_worker, drop_last=True)
val_loader = torch.utils.data.DataLoader(dataset_val, pin_memory=True, num_workers=12, shuffle=False,
                                            drop_last=False,worker_init_fn=seed_worker,batch_size=args.val_bs)

# Attempt to deal with data imbalance: inverse frequency divided by lowest frequency class (0.5 < class_weight <= 1)
if distr is not None:
    args.ce_weights = (torch.tensor([(sum(distr) / freq) / (sum(distr) / min(distr)) for freq in distr]).float()
                        .to(device)) * 0.5 + 0.5
else:
    args.ce_weights = None

# Set log intervals
args.log_interval = args.log_interval // args.batch_size
args.val_interval = args.val_interval // args.batch_size

use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"Using {'GPU.' if use_cuda else 'CPU, as was explicitly requested, or as GPU is not available.'}")

# Try and get the dataset distribution and lookup table
try:
    distr = dataset_train.get_distribution()
    print(f"Training dataset size: {sum(distr)}, with {len(distr)} classes and distribution: {distr}")
    print(f"Validation dataset size: {sum(dataset_val.get_distribution())}, "
            f"with distribution: {dataset_val.get_distribution()}")

    lookup_table = dataset_train.get_lookup_table()
except:
    print("An error occurred while getting the dataset distribution and lookup table. Likely the dataset does not "
            "have an implementation for these functions.")
    



Using GPU.
no look up table found
look up table found
Loaded dataset: gmdb (version v1.1.0) with image size 112x112 in RGB, while not retaining the aspect ratio.
Using GPU.
Training dataset size: 7787, with 275 classes and distribution: [361, 241, 224, 171, 168, 147, 144, 132, 131, 119, 111, 96, 93, 93, 84, 75, 75, 74, 73, 72, 71, 69, 67, 63, 63, 62, 61, 56, 54, 54, 54, 51, 51, 50, 49, 47, 47, 47, 45, 45, 44, 44, 43, 42, 42, 42, 42, 42, 42, 40, 39, 39, 38, 38, 37, 36, 36, 36, 35, 35, 35, 34, 33, 33, 31, 31, 31, 31, 30, 30, 30, 30, 29, 29, 29, 28, 28, 28, 28, 27, 27, 27, 27, 27, 26, 26, 26, 26, 26, 26, 26, 25, 25, 24, 24, 24, 24, 24, 24, 23, 23, 23, 23, 23, 23, 22, 22, 22, 21, 21, 21, 21, 21, 21, 21, 21, 21, 20, 20, 20, 20, 19, 19, 19, 19, 19, 19, 19, 18, 18, 18, 18, 18, 18, 18, 17, 17, 17, 17, 17, 17, 17, 17, 17, 16, 16, 16, 16, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 14, 14, 14, 13, 13, 13, 13, 13, 13, 13, 12, 12, 12, 12, 12, 12, 12, 12, 12, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,

In [ ]:
device='cuda'
def entropy(p, eps=1e-12):
    p = torch.clamp(p, eps, 1.0)
    return -(p * p.log()).sum(dim=-1) # sum over last dim (classes)
def enable_dropout(m):  # set all dropout layers to train mode
    for mod in m.modules():
        if mod.__class__.__name__.startswith('Dropout'):
            mod.train()

def mc_dropout_mi(model, x, device=device, T=20, batch_size=128):
    """
    Monte Carlo Dropout Mutual Information (BALD-style epistemic uncertainty)
    with safe batch-wise inference
    """
    model.eval()
    enable_dropout(model)  # activate dropout layers

    ps = []
    with torch.no_grad():
        for _ in range(T):
            batch_preds = []
            for i in range(0, len(x), batch_size):
                xb = x[i:i+batch_size].to(device, dtype=torch.float32)
                logits, _ = model(xb.to(device, dtype=torch.float32))
                batch_preds.append(F.softmax(logits, dim=1).cpu())
            ps.append(torch.cat(batch_preds, dim=0))  # [B,C] for each T
    # [T,B,C]
    P = torch.stack(ps, dim=0)

    # Mean prediction across T stochastic passes
    p_bar = P.mean(dim=0)  # [B,C]

    # Entropy of mean prediction (total uncertainty)
    H_bar = entropy(p_bar)

    # Mean entropy (expected data uncertainty) entropy(P) goes over B that means entropy of class distribution, then mean over all classes and then dropouts
    H_exp = entropy(P).mean(dim=0)#.mean()

    # Mutual Information (epistemic uncertainty)
    MI = H_bar - H_exp

    return MI, H_bar, H_exp, p_bar

# Swarm models
model1 = torch.load('s1_glint360k_r50_512d_gmdb__v1.1.0_bs64_size112_channels3_swarm_model_epoch_49_Oct_2025.pth').to(device)  # swarm ResNet-50
model2 = torch.load('s1_glint360k_r100_512d_gmdb__v1.1.0_bs128_size112_channels3_swarm_model_epoch_49_Oct_2025.pth').to(device) # swarm ResNet-100 
# model2 = torch.load('s1_glint360k_r100_512d_gmdb_aidy__v1.1.0_bs128_size112_channels3_swarm_model.pth').to(device) # swarm ResNet-100 GMDB+AIDY

## central models
model3 = torch.load('s1_glint360k_r50_512d_gmdb__v1.1.0_bs64_size112_channels3_last_model.pth') # GMDB central ResNet-50
model4 = torch.load('s2_glint360k_r100_512d_gmdb__v1.1.0_bs128_size112_channels3_last_model.pth')  # GMDB central ResNet-100
# model4 = torch.load('s1_glint360k_r100_512d_gmdb_aidy__v1.1.0_bs128_size112_channels3_swarm_model_epoch_49_central.pth').to(device) # GMDB+AIDY central 


from itertools import chain

combined_loader = chain(train_loader, val_loader)

all_x = []
all_y = []

with torch.no_grad():
    for x, y in val_loader:  
        all_x.append(x)
        all_y.append(y)

# Concatenate into single tensors
all_x = torch.cat(all_x, dim=0)  # Shape: [num_samples, channels, height, width]
all_y = torch.cat(all_y, dim=0)  # Shape: [num_samples] or [num_samples, num_classes]

MI_swarm_res_50, H_bar_swarm_res_50, H_exp_swarm_res_50, p_bar_swarm_res_50 = mc_dropout_mi(model1, all_x)
MI_swarm_res_100, H_bar_swarm_res_100, H_exp_swarm_res_100, p_bar_swarm_res_100 = mc_dropout_mi(model2, all_x)

MI_central_res_50, H_bar_central_res_50, H_exp_central_res_50, p_bar_central_res_50 = mc_dropout_mi(model3, all_x)
MI_central_res_100, H_bar_central_res_100, H_exp_central_res_100, p_bar_central_res_100 = mc_dropout_mi(model4, all_x)



aleatoric_swarm_res_50 = H_bar_swarm_res_50 - MI_swarm_res_50
aleatoric_swarm_res_100 = H_bar_swarm_res_100 - MI_swarm_res_100
aleatoric_central_res_50 = H_bar_central_res_50 - MI_central_res_50
aleatoric_central_res_100 = H_bar_central_res_100 - MI_central_res_100

# Safety clamp to avoid tiny negative values due to float precision
aleatoric_swarm_res_50 = torch.clamp(aleatoric_swarm_res_50, min=0.0)
aleatoric_swarm_res_100 = torch.clamp(aleatoric_swarm_res_100, min=0.0)
aleatoric_central_res_50 = torch.clamp(aleatoric_central_res_50, min=0.0)
aleatoric_central_res_100 = torch.clamp(aleatoric_central_res_100, min=0.0)


unique_elements = all_y.unique()
unique_elements = unique_elements.unsqueeze(1)
unique_elements
matches = (all_y == unique_elements)

MI_per_class_swarm_res_50 = [MI_swarm_res_50[i].cpu().numpy() for i in matches] # len(all_y[all_y==3])
H_bar_per_class_swarm_res_50 = [H_bar_swarm_res_50[i].cpu().numpy() for i in matches]

MI_per_class_swarm_res_100 = [MI_swarm_res_100[i].cpu().numpy() for i in matches] # len(all_y[all_y==3])
H_bar_per_class_swarm_res_100 = [H_bar_swarm_res_100[i].cpu().numpy() for i in matches]


MI_per_class_central_res_50 = [MI_central_res_50[i].cpu().numpy() for i in matches] # len(all_y[all_y==3])
H_bar_per_class_central_res_50 = [H_bar_central_res_50[i].cpu().numpy() for i in matches]

MI_per_class_central_res_100 = [MI_central_res_100[i].cpu().numpy() for i in matches] # len(all_y[all_y==3])
H_bar_per_class_central_res_100 = [H_bar_central_res_100[i].cpu().numpy() for i in matches]


AU_per_class_swarm_res_50 = [aleatoric_swarm_res_50[i].cpu().numpy() for i in matches]
AU_per_class_swarm_res_100 = [aleatoric_swarm_res_100[i].cpu().numpy() for i in matches]

AU_per_class_central_res_50 = [aleatoric_central_res_50[i].cpu().numpy() for i in matches]
AU_per_class_central_res_100 = [aleatoric_central_res_100[i].cpu().numpy() for i in matches]


MI_flat_swarm_res_50 = [np.ravel(arr) for arr in MI_per_class_swarm_res_50]
AU_flat_swarm_res_50 = [np.ravel(arr) for arr in AU_per_class_swarm_res_50]
MI_flat_swarm_res_100 = [np.ravel(arr) for arr in MI_per_class_swarm_res_100]
AU_flat_swarm_res_100 = [np.ravel(arr) for arr in AU_per_class_swarm_res_100]

MI_flat_central_res_50 = [np.ravel(arr) for arr in MI_per_class_central_res_50]
AU_flat_central_res_50 = [np.ravel(arr) for arr in AU_per_class_central_res_50]
MI_flat_central_res_100 = [np.ravel(arr) for arr in MI_per_class_central_res_100]
AU_flat_central_res_100 = [np.ravel(arr) for arr in AU_per_class_central_res_100]

MI_mean_swarm_res_50 = np.array([np.mean(np.ravel(arr)) for arr in MI_flat_swarm_res_50])
AU_mean_swarm_res_50 = np.array([np.mean(np.ravel(arr)) for arr in AU_flat_swarm_res_50])
MI_mean_swarm_res_100 = np.array([np.mean(np.ravel(arr)) for arr in MI_flat_swarm_res_100])
AU_mean_swarm_res_100 = np.array([np.mean(np.ravel(arr)) for arr in AU_flat_swarm_res_100])

MI_mean_central_res_50 = np.array([np.mean(np.ravel(arr)) for arr in MI_flat_central_res_50])
AU_mean_central_res_50 = np.array([np.mean(np.ravel(arr)) for arr in AU_flat_central_res_50])
MI_mean_central_res_100 = np.array([np.mean(np.ravel(arr)) for arr in MI_flat_central_res_100])
AU_mean_central_res_100 = np.array([np.mean(np.ravel(arr)) for arr in AU_flat_central_res_100])

In [18]:
import plotly.graph_objects as go
import numpy as np

class_dist = np.array([len(all_y[i]) for i in matches])

arr1 = np.asarray(MI_mean_swarm_res_50) / class_dist
arr2 = np.asarray(MI_mean_central_res_50) / class_dist

x = np.arange(len(arr1))

fig = go.Figure()

# ---------------------------------------------------------
# 1. Add connecting line for each class
#    Blue  = Swarm is lower
#    Red   = Central is lower
# ---------------------------------------------------------

for i in range(len(x)):

    if arr1[i] < arr2[i]:
        line_color = 'blue'   # Swarm lower
    else:
        line_color = 'red'    # Central lower

    fig.add_trace(
        go.Scatter(
            x=[x[i], x[i]],
            y=[arr1[i], arr2[i]],
            mode='lines',
            line=dict(
                color=line_color,
                width=3
            ),
            opacity=0.5,
            showlegend=False,
            hoverinfo='skip'
        )
    )

# ---------------------------------------------------------
# 2. Swarm model dots
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x,
        y=arr1,
        mode='markers',
        name='Swarm Resnet 50',
        marker=dict(
            color='blue',
            size=7
        ),
        opacity=0.5,
        hovertemplate=(
            'Class: %{x}<br>'
            'Swarm: %{y:.4f}<extra></extra>'
        )
    )
)

# ---------------------------------------------------------
# 3. Central model dots
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x,
        y=arr2,
        mode='markers',
        name='Central Resnet 50',
        marker=dict(
            color='red',
            size=7
        ),
        opacity=0.5,
        hovertemplate=(
            'Class: %{x}<br>'
            'Central: %{y:.4f}<extra></extra>'
        )
    )
)
# Add new trace with right-hand side axis
fig.add_trace(go.Scatter(
    x=x, 
    y=np.log(class_dist), 
    mode='markers', 
    name='', 
    line=dict(color='#009E73'), 
    yaxis='y3',
    showlegend=False
))
# ---------------------------------------------------------
# 4. Layout
# ---------------------------------------------------------

fig.update_layout(
    height=500,
    width=1200,

    margin=dict(
        l=60,
        r=20,
        t=20,
        b=60
    ),

    paper_bgcolor='white',
    plot_bgcolor='white',

    xaxis=dict(
        title='Classes',
        title_font=dict(
            family='Verdana',
            size=20
        ),
        tickfont=dict(size=14),
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        title='MI: BALD-Value per class',
        title_font=dict(
            family='Verdana',
            size=18
        ),
        tickfont=dict(size=14),
        showgrid=False,
        zeroline=True
    ),
    yaxis3=dict(
        title="No. of Samples",       # label for right axis
        title_font=dict(
            family='Verdana',
            size=18
        ),
        tickfont=dict(size=14),
        overlaying='y',         # share the same x-axis
        side='right',           # right side of top plot
        anchor='x',             # align with same x
        position=1.0            # ensure it's on far right
    ),

    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        font=dict(
            size=18,
            color='black'
        )
    )
)

fig.show()


In [17]:
import plotly.graph_objects as go
import numpy as np

class_dist = np.array([len(all_y[i]) for i in matches])

arr1 = np.asarray(MI_mean_swarm_res_100) / class_dist
arr2 = np.asarray(MI_mean_central_res_100) / class_dist

x = np.arange(len(arr1))

fig = go.Figure()

# ---------------------------------------------------------
# 1. Add connecting line for each class
#    Blue  = Swarm is lower
#    Red   = Central is lower
# ---------------------------------------------------------

for i in range(len(x)):

    if arr1[i] < arr2[i]:
        line_color = 'blue'   # Swarm lower
    else:
        line_color = 'red'    # Central lower

    fig.add_trace(
        go.Scatter(
            x=[x[i], x[i]],
            y=[arr1[i], arr2[i]],
            mode='lines',
            line=dict(
                color=line_color,
                width=3
            ),
            opacity=0.5,
            showlegend=False,
            hoverinfo='skip'
        )
    )

# ---------------------------------------------------------
# 2. Swarm model dots
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x,
        y=arr1,
        mode='markers',
        name='Swarm Resnet 100',
        marker=dict(
            color='blue',
            size=7
        ),
        opacity=0.5,
        hovertemplate=(
            'Class: %{x}<br>'
            'Swarm: %{y:.4f}<extra></extra>'
        )
    )
)

# ---------------------------------------------------------
# 3. Central model dots
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x,
        y=arr2,
        mode='markers',
        name='Central Resnet 100',
        marker=dict(
            color='red',
            size=7
        ),
        opacity=0.5,
        hovertemplate=(
            'Class: %{x}<br>'
            'Central: %{y:.4f}<extra></extra>'
        )
    )
)
# Add new trace with right-hand side axis
fig.add_trace(go.Scatter(
    x=x, 
    y=np.log(class_dist), 
    mode='markers', 
    name='', 
    line=dict(color='#009E73'), 
    yaxis='y3',
    showlegend=False
))
# ---------------------------------------------------------
# 4. Layout
# ---------------------------------------------------------

fig.update_layout(
    height=500,
    width=1200,

    margin=dict(
        l=60,
        r=20,
        t=20,
        b=60
    ),

    paper_bgcolor='white',
    plot_bgcolor='white',

    xaxis=dict(
        title='Classes',
        title_font=dict(
            family='Verdana',
            size=20
        ),
        tickfont=dict(size=14),
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        title='MI: BALD-Value per class',
        title_font=dict(
            family='Verdana',
            size=18
        ),
        tickfont=dict(size=14),
        showgrid=False,
        zeroline=True
    ),
    yaxis3=dict(
        title="No. of Samples",       # label for right axis
        title_font=dict(
            family='Verdana',
            size=18
        ),
        tickfont=dict(size=14),
        overlaying='y',         # share the same x-axis
        side='right',           # right side of top plot
        anchor='x',             # align with same x
        position=1.0            # ensure it's on far right
    ),

    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        font=dict(
            size=18,
            color='black'
        )
    )
)

fig.show()
